<a href="https://colab.research.google.com/github/saiprasadag20/STTP_on_ml/blob/main/Employment_Mlmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Default title text
import pandas as pd
from sklearn.metrics import accuracy_score, mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans

url = "https://raw.githubusercontent.com/m0rningLight/Data_Analysis--Layoffs_Dataset/refs/heads/main/data/layoffs_raw.csv"
df = pd.read_csv(url)
df_original = df.copy()
df.columns = df.columns.str.lower().str.strip()
df = df.drop_duplicates()
df['industry'] = df['industry'].fillna("unknown")
df['country'] = df['country'].fillna("unknown")
df = df.fillna(0)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year
df = df.dropna()

def classify(x):
    if x > 1000:
        return "high"
    elif x > 100:
        return "medium"
    else:
        return "low"

df['layoff_category'] = df['total_laid_off'].apply(classify)
le = LabelEncoder()
df['country_encoded'] = le.fit_transform(df['country'])
df['industry'] = le.fit_transform(df['industry'])
df['country'] = le.fit_transform(df['country'])

X = df[['industry', 'country', 'funds_raised_millions', 'year']]
y_reg = df['total_laid_off']
y_clf = df['layoff_category']

X_train, X_test, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
_, _, y_train_clf, y_test_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# LINEAR REGRESSION

lr = LinearRegression()
lr.fit(X_train, y_train_reg)
pred_reg = lr.predict(X_test)
mae = mean_absolute_error(y_test_reg, pred_reg)
print("Linear Regression Sample Output:", pred_reg[:5])

# KNN CLASSIFICATION

knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_scaled, y_train_clf)
pred_clf = knn.predict(X_test_scaled)
acc = accuracy_score(y_test_clf, pred_clf)
print("KNN Sample Output:", pred_clf[:5])

# K-MEANS CLUSTERING

kmeans = KMeans(n_clusters=3, random_state=42)
df['cluster'] = kmeans.fit_predict(X)
print(df[['industry','cluster']].head(25))

# COMPARISON TABLE

results = X_test.copy()

results['actual_layoffs'] = y_test_reg.values
results['predicted_layoffs'] = pred_reg

results['actual_category'] = y_test_clf.values
results['predicted_category'] = pred_clf

results['cluster'] = kmeans.predict(X_test)

print("\nComparison Table:")
print(results.head(25))

# GRAPH: LAYOFFS TREND

layoffs_year = df.groupby('year')['total_laid_off'].sum()

plt.figure()
layoffs_year.plot()
plt.title("Year-wise Layoffs Trend")
plt.xlabel("Year")
plt.ylabel("Total Layoffs")
plt.show()

# COMPANY-WISE LAYOFFS

company_layoffs = df.groupby('company')['total_laid_off'].sum()
company_layoffs = company_layoffs.sort_values(ascending=False)
top_companies = company_layoffs.head(10)
plt.figure()
top_companies.plot(kind='bar')
plt.title("Top 10 Companies by Layoffs")
plt.xlabel("Company")
plt.ylabel("Total Layoffs")
plt.xticks(rotation=45)
plt.show()

#Country Wise Layoffs


df_country = df_original[df_original['country'].isin(['India','Netherlands','United Arab Emirates',
                                                      'United Kingdom','United States'])]
country_layoffs = df_country.groupby('country')['total_laid_off'].sum()
country_layoffs.plot(kind='bar',color=['orange','white','green','red','blue'])
plt.show()